In [1]:
import os

print(os.getcwd())

c:\Users\Greesha Vaishnavi\Desktop\dsprojects\Hotel_Booking_Cancellation_Prediction\research\model_experiments


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    log_loss
) 

In [3]:
import pandas as pd

train_df = pd.read_csv(
    "../../artifacts/data_transformation/train.csv"
)

test_df = pd.read_csv(
    "../../artifacts/data_transformation/test.csv"
)

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

Train shape: (95512, 917)
Test shape : (23878, 917)


In [4]:
target_column = "is_canceled"

X_train = train_df.drop(
    columns=[target_column]
)

y_train = train_df[target_column]

X_test = test_df.drop(
    columns=[target_column]
)

y_test = test_df[target_column]

In [5]:
print("X_train shape:", X_train.shape)  #check our transformed data
print("X_test shape :", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape :", y_test.shape)

X_train shape: (95512, 916)
X_test shape : (23878, 916)
y_train shape: (95512,)
y_test shape : (23878,)


# Baseline model

In [6]:
random_forest_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

In [7]:
random_forest_model.fit(
    X_train,
    y_train
)

print("Random Forest training completed.")

Random Forest training completed.


In [8]:
rf_train_pred = random_forest_model.predict(
    X_train
)

rf_test_pred = random_forest_model.predict(
    X_test
)

rf_train_accuracy = accuracy_score(
    y_train,
    rf_train_pred
)

rf_test_accuracy = accuracy_score(
    y_test,
    rf_test_pred
)

print("Training Accuracy:", rf_train_accuracy)
print("Testing Accuracy :", rf_test_accuracy)

Training Accuracy: 0.9963250691012647
Testing Accuracy : 0.8922857860792361


In [9]:
rf_pred = random_forest_model.predict(
    X_test
)

rf_prob = random_forest_model.predict_proba(
    X_test
)[:, 1]

In [10]:
rf_cm = confusion_matrix(
    y_test,
    rf_pred
)

tn_rf, fp_rf, fn_rf, tp_rf = rf_cm.ravel()

In [11]:
rf_accuracy = accuracy_score(
    y_test,
    rf_pred
)

rf_precision = precision_score(
    y_test,
    rf_pred,
    zero_division=0
)

rf_recall = recall_score(
    y_test,
    rf_pred,
    zero_division=0
)

rf_specificity = (
    tn_rf / (tn_rf + fp_rf)
)

rf_f1 = f1_score(
    y_test,
    rf_pred,
    zero_division=0
)

rf_roc_auc = roc_auc_score(
    y_test,
    rf_prob
)

rf_pr_auc = average_precision_score(
    y_test,
    rf_prob
)

rf_logloss = log_loss(
    y_test,
    rf_prob
)

In [12]:
print("=" * 55)
print("RANDOM FOREST - BASELINE EVALUATION")
print("=" * 55)

print(f"Accuracy    : {rf_accuracy:.4f}")
print(f"Precision   : {rf_precision:.4f}")
print(f"Recall      : {rf_recall:.4f}")
print(f"Specificity : {rf_specificity:.4f}")
print(f"F1 Score    : {rf_f1:.4f}")
print(f"ROC-AUC     : {rf_roc_auc:.4f}")
print(f"PR-AUC      : {rf_pr_auc:.4f}")
print(f"Log Loss    : {rf_logloss:.4f}")

RANDOM FOREST - BASELINE EVALUATION
Accuracy    : 0.8923
Precision   : 0.8953
Recall      : 0.8032
Specificity : 0.9447
F1 Score    : 0.8467
ROC-AUC     : 0.9569
PR-AUC      : 0.9394
Log Loss    : 0.2739


In [13]:
n_estimators_values = [50, 100, 200, 300]    # n_estimator values

In [14]:
n_estimators_results = []

for n_trees in n_estimators_values:

    model = RandomForestClassifier(
        n_estimators=n_trees,
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train,
        y_train
    )

    train_pred = model.predict(X_train)

    test_pred = model.predict(X_test)

    test_prob = model.predict_proba(
        X_test
    )[:, 1]

    # Confusion Matrix
    cm = confusion_matrix(
        y_test,
        test_pred
    )

    tn, fp, fn, tp = cm.ravel()

    # 8 metrics

    accuracy_value = accuracy_score(
        y_test,
        test_pred
    )

    precision_value = precision_score(
        y_test,
        test_pred,
        zero_division=0
    )

    recall_value = recall_score(
        y_test,
        test_pred,
        zero_division=0
    )

    specificity_value = (
        tn / (tn + fp)
    )

    f1_value = f1_score(
        y_test,
        test_pred,
        zero_division=0
    )

    roc_auc_value = roc_auc_score(
        y_test,
        test_prob
    )

    pr_auc_value = average_precision_score(
        y_test,
        test_prob
    )

    logloss_value = log_loss(
        y_test,
        test_prob
    )

    n_estimators_results.append({
        "n_estimators": n_trees,

        "Train Accuracy": accuracy_score(
            y_train,
            train_pred
        ),

        "Accuracy": accuracy_value,
        "Precision": precision_value,
        "Recall": recall_value,
        "Specificity": specificity_value,
        "F1 Score": f1_value,
        "ROC-AUC": roc_auc_value,
        "PR-AUC": pr_auc_value,
        "Log Loss": logloss_value
    })

In [15]:
n_estimators_results_df = pd.DataFrame(
    n_estimators_results
)

n_estimators_results_df.round(4)

,n_estimators,Train Accuracy,Accuracy,Precision,Recall,Specificity,F1 Score,ROC-AUC,PR-AUC,Log Loss
0,50,0.9962,0.8907,0.8945,0.7992,0.9445,0.8442,0.9556,0.9367,0.2819
1,100,0.9963,0.8923,0.8953,0.8032,0.9447,0.8467,0.9569,0.9394,0.2739
2,200,0.9963,0.8928,0.8968,0.8031,0.9457,0.8474,0.9580,0.9410,0.2660
3,300,0.9963,0.8929,0.8961,0.8042,0.9451,0.8476,0.9583,0.9416,0.2652


In [16]:
max_depth_values = [5, 10, 15, 20, 25, None]   # max_depth_values

In [17]:
max_depth_results = []

for depth in max_depth_values:

    model = RandomForestClassifier(
        n_estimators=300,
        max_depth=depth,
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train,
        y_train
    )

    train_pred = model.predict(X_train)

    test_pred = model.predict(X_test)

    test_prob = model.predict_proba(
        X_test
    )[:, 1]

    # Confusion Matrix
    cm = confusion_matrix(
        y_test,
        test_pred
    )

    tn, fp, fn, tp = cm.ravel()

    # 8 metrics

    accuracy_value = accuracy_score(
        y_test,
        test_pred
    )

    precision_value = precision_score(
        y_test,
        test_pred,
        zero_division=0
    )

    recall_value = recall_score(
        y_test,
        test_pred,
        zero_division=0
    )

    specificity_value = (
        tn / (tn + fp)
    )

    f1_value = f1_score(
        y_test,
        test_pred,
        zero_division=0
    )

    roc_auc_value = roc_auc_score(
        y_test,
        test_prob
    )

    pr_auc_value = average_precision_score(
        y_test,
        test_prob
    )

    logloss_value = log_loss(
        y_test,
        test_prob
    )

    max_depth_results.append({
        "max_depth": depth,

        "Train Accuracy": accuracy_score(
            y_train,
            train_pred
        ),

        "Accuracy": accuracy_value,
        "Precision": precision_value,
        "Recall": recall_value,
        "Specificity": specificity_value,
        "F1 Score": f1_value,
        "ROC-AUC": roc_auc_value,
        "PR-AUC": pr_auc_value,
        "Log Loss": logloss_value
    })

In [18]:
max_depth_results_df = pd.DataFrame(
    max_depth_results
)

max_depth_results_df.round(4)

,max_depth,Train Accuracy,Accuracy,Precision,Recall,Specificity,F1 Score,ROC-AUC,PR-AUC,Log Loss
0,5.0,0.7601,0.7586,0.9994,0.3487,0.9999,0.5170,0.8854,0.8509,0.5344
1,10.0,0.7714,0.7702,0.9875,0.3845,0.9971,0.5535,0.9150,0.8861,0.4570
2,15.0,0.8126,0.8049,0.9710,0.4880,0.9914,0.6495,0.9318,0.9078,0.4015
3,20.0,0.8738,0.8526,0.9310,0.6503,0.9717,0.7658,0.9436,0.9221,0.3565
4,25.0,0.9129,0.8726,0.9187,0.7197,0.9625,0.8072,0.9516,0.9324,0.3209
5,NaN,0.9963,0.8929,0.8961,0.8042,0.9451,0.8476,0.9583,0.9416,0.2652


In [19]:
split_values = [2, 5, 10, 20, 50, 100]   # min_samples_splits

In [20]:
split_results = []

for min_split in split_values:

    model = RandomForestClassifier(
        n_estimators=100,
        max_depth=None,
        min_samples_split=min_split,
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train,
        y_train
    )

    train_pred = model.predict(X_train)

    test_pred = model.predict(X_test)

    test_prob = model.predict_proba(
        X_test
    )[:, 1]

    # Confusion Matrix
    cm = confusion_matrix(
        y_test,
        test_pred
    )

    tn, fp, fn, tp = cm.ravel()

    # 8 metrics

    accuracy_value = accuracy_score(
        y_test,
        test_pred
    )

    precision_value = precision_score(
        y_test,
        test_pred,
        zero_division=0
    )

    recall_value = recall_score(
        y_test,
        test_pred,
        zero_division=0
    )

    specificity_value = (
        tn / (tn + fp)
    )

    f1_value = f1_score(
        y_test,
        test_pred,
        zero_division=0
    )

    roc_auc_value = roc_auc_score(
        y_test,
        test_prob
    )

    pr_auc_value = average_precision_score(
        y_test,
        test_prob
    )

    logloss_value = log_loss(
        y_test,
        test_prob
    )

    split_results.append({
        "min_samples_split": min_split,

        "Train Accuracy": accuracy_score(
            y_train,
            train_pred
        ),

        "Accuracy": accuracy_value,
        "Precision": precision_value,
        "Recall": recall_value,
        "Specificity": specificity_value,
        "F1 Score": f1_value,
        "ROC-AUC": roc_auc_value,
        "PR-AUC": pr_auc_value,
        "Log Loss": logloss_value
    })

In [ ]:
# # Display results

split_results_df = pd.DataFrame( split_results)

split_results_df.round(4)

,min_samples_split,Train Accuracy,Accuracy,Precision,Recall,Specificity,F1 Score,ROC-AUC,PR-AUC,Log Loss
0,2,0.9963,0.8923,0.8953,0.8032,0.9447,0.8467,0.9569,0.9394,0.2739
1,5,0.9804,0.8893,0.8988,0.7901,0.9476,0.8409,0.9574,0.9402,0.2701
2,10,0.9511,0.8860,0.9025,0.7761,0.9506,0.8345,0.9566,0.9381,0.2779
3,20,0.9240,0.8801,0.9020,0.7586,0.9515,0.8241,0.9544,0.9341,0.2896
4,50,0.8949,0.8706,0.9043,0.7278,0.9547,0.8065,0.9495,0.9273,0.3103
5,100,0.8803,0.8628,0.9067,0.7019,0.9575,0.7912,0.9458,0.9222,0.3258


In [22]:
leaf_values = [1, 2, 5, 10, 20, 50]   # min_samples_leaf

In [23]:
leaf_results = []

for min_leaf in leaf_values:

    model = RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_split=5,
        min_samples_leaf=min_leaf,
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train,
        y_train
    )

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    test_prob = model.predict_proba(
        X_test
    )[:, 1]

    # Confusion Matrix
    cm = confusion_matrix(
        y_test,
        test_pred
    )

    tn, fp, fn, tp = cm.ravel()

    # 8 metrics
    accuracy_value = accuracy_score(
        y_test,
        test_pred
    )

    precision_value = precision_score(
        y_test,
        test_pred,
        zero_division=0
    )

    recall_value = recall_score(
        y_test,
        test_pred,
        zero_division=0
    )

    specificity_value = tn / (tn + fp)

    f1_value = f1_score(
        y_test,
        test_pred,
        zero_division=0
    )

    roc_auc_value = roc_auc_score(
        y_test,
        test_prob
    )

    pr_auc_value = average_precision_score(
        y_test,
        test_prob
    )

    logloss_value = log_loss(
        y_test,
        test_prob
    )

    leaf_results.append({
        "min_samples_leaf": min_leaf,

        "Train Accuracy": accuracy_score(
            y_train,
            train_pred
        ),

        "Accuracy": accuracy_value,
        "Precision": precision_value,
        "Recall": recall_value,
        "Specificity": specificity_value,
        "F1 Score": f1_value,
        "ROC-AUC": roc_auc_value,
        "PR-AUC": pr_auc_value,
        "Log Loss": logloss_value
    })

In [24]:
leaf_results_df = pd.DataFrame(
    leaf_results
)

leaf_results_df.round(4)

,min_samples_leaf,Train Accuracy,Accuracy,Precision,Recall,Specificity,F1 Score,ROC-AUC,PR-AUC,Log Loss
0,1,0.9811,0.8919,0.9012,0.7954,0.9487,0.8450,0.9589,0.9417,0.2679
1,2,0.8961,0.8676,0.9142,0.7091,0.9608,0.7987,0.9501,0.9294,0.3218
2,5,0.8594,0.8504,0.9227,0.6508,0.9679,0.7632,0.9403,0.9168,0.3584
3,10,0.8387,0.8333,0.9323,0.5931,0.9747,0.7250,0.9327,0.9076,0.3834
4,20,0.8202,0.8164,0.9466,0.5347,0.9822,0.6833,0.9275,0.9011,0.4030
5,50,0.7872,0.7849,0.9732,0.4313,0.9930,0.5977,0.9178,0.8905,0.4342


In [25]:
max_features_values = ["sqrt", "log2", None]   # max_features

In [26]:
max_features_results = []

for max_feat in max_features_values:

    model = RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_split=5,
        min_samples_leaf=1,
        max_features=max_feat,
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train,
        y_train
    )

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    test_prob = model.predict_proba(
        X_test
    )[:, 1]

    # Confusion Matrix
    cm = confusion_matrix(
        y_test,
        test_pred
    )

    tn, fp, fn, tp = cm.ravel()

    # 8 metrics

    accuracy_value = accuracy_score(
        y_test,
        test_pred
    )

    precision_value = precision_score(
        y_test,
        test_pred,
        zero_division=0
    )

    recall_value = recall_score(
        y_test,
        test_pred,
        zero_division=0
    )

    specificity_value = tn / (tn + fp)

    f1_value = f1_score(
        y_test,
        test_pred,
        zero_division=0
    )

    roc_auc_value = roc_auc_score(
        y_test,
        test_prob
    )

    pr_auc_value = average_precision_score(
        y_test,
        test_prob
    )

    logloss_value = log_loss(
        y_test,
        test_prob
    )

    max_features_results.append({
        "max_features": max_feat,

        "Train Accuracy": accuracy_score(
            y_train,
            train_pred
        ),

        "Accuracy": accuracy_value,
        "Precision": precision_value,
        "Recall": recall_value,
        "Specificity": specificity_value,
        "F1 Score": f1_value,
        "ROC-AUC": roc_auc_value,
        "PR-AUC": pr_auc_value,
        "Log Loss": logloss_value
    })

In [27]:
max_features_results_df = pd.DataFrame(
    max_features_results
)

max_features_results_df.round(4)

,max_features,Train Accuracy,Accuracy,Precision,Recall,Specificity,F1 Score,ROC-AUC,PR-AUC,Log Loss
0,sqrt,0.9811,0.8919,0.9012,0.7954,0.9487,0.8450,0.9589,0.9417,0.2679
1,log2,0.9798,0.8850,0.9085,0.7669,0.9546,0.8317,0.9551,0.9377,0.2857
2,None,0.9917,0.8920,0.8700,0.8329,0.9268,0.8510,0.9596,0.9410,0.2693


# Hyperparameter Tuning

In [28]:
from sklearn.model_selection import GridSearchCV

In [29]:
rf_param_grid = {
    "n_estimators": [200],
    "max_depth": [None],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1],
    "max_features": ["sqrt", None]
}

In [30]:
rf_grid_search = GridSearchCV(
    estimator=RandomForestClassifier(
        random_state=42,
        n_jobs=1
    ),
    param_grid=rf_param_grid,
    cv=3,
    scoring="f1",
    n_jobs=-1,
    verbose=1
)

In [31]:
rf_grid_search.fit(
    X_train,
    y_train
)

Fitting 3 folds for each of 4 candidates, totalling 12 fits


,estimator,RandomForestC...ndom_state=42)
,param_grid,"{'max_depth': [None], 'max_features': ['sqrt', None], 'min_samples_leaf': [1], 'min_samples_split': [2, 5], ...}"
,scoring,'f1'
,n_jobs,-1
,refit,True
,cv,3
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,200


In [44]:
print("GridSearchCV completed successfully.")
print("Best Parameters:")
print(rf_grid_search.best_params_)

GridSearchCV completed successfully.
Best Parameters:
{'max_depth': None, 'max_features': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}


In [45]:
print(
    "Best CV F1 Score:",
    rf_grid_search.best_score_
)

Best CV F1 Score: 0.8463122235656012


In [46]:
best_rf_model = rf_grid_search.best_estimator_

In [47]:
print("Best Number of Trees:",
      best_rf_model.n_estimators)

print("Best Max Depth:",
      best_rf_model.max_depth)

print("Best Min Samples Split:",
      best_rf_model.min_samples_split)

print("Best Min Samples Leaf:",
      best_rf_model.min_samples_leaf)

print("Best Max Features:",
      best_rf_model.max_features)

Best Number of Trees: 200
Best Max Depth: None
Best Min Samples Split: 2
Best Min Samples Leaf: 1
Best Max Features: None


In [48]:
rf_tuned_pred = best_rf_model.predict(
    X_test
)

rf_tuned_prob = best_rf_model.predict_proba(
    X_test
)[:, 1]

In [49]:
rf_tuned_cm = confusion_matrix(
    y_test,
    rf_tuned_pred
)

tn_rf_tuned, fp_rf_tuned, fn_rf_tuned, tp_rf_tuned = (
    rf_tuned_cm.ravel()
)

In [50]:
rf_tuned_accuracy = accuracy_score(
    y_test,
    rf_tuned_pred
)

rf_tuned_precision = precision_score(
    y_test,
    rf_tuned_pred,
    zero_division=0
)

rf_tuned_recall = recall_score(
    y_test,
    rf_tuned_pred,
    zero_division=0
)

rf_tuned_specificity = (
    tn_rf_tuned /
    (tn_rf_tuned + fp_rf_tuned)
)

rf_tuned_f1 = f1_score(
    y_test,
    rf_tuned_pred,
    zero_division=0
)

rf_tuned_roc_auc = roc_auc_score(
    y_test,
    rf_tuned_prob
)

rf_tuned_pr_auc = average_precision_score(
    y_test,
    rf_tuned_prob
)

rf_tuned_logloss = log_loss(
    y_test,
    rf_tuned_prob
)

In [51]:
print("=" * 55)
print("RANDOM FOREST - TUNED MODEL EVALUATION")
print("=" * 55)

print(f"Accuracy    : {rf_tuned_accuracy:.4f}")
print(f"Precision   : {rf_tuned_precision:.4f}")
print(f"Recall      : {rf_tuned_recall:.4f}")
print(f"Specificity : {rf_tuned_specificity:.4f}")
print(f"F1 Score    : {rf_tuned_f1:.4f}")
print(f"ROC-AUC     : {rf_tuned_roc_auc:.4f}")
print(f"PR-AUC      : {rf_tuned_pr_auc:.4f}")
print(f"Log Loss    : {rf_tuned_logloss:.4f}")

RANDOM FOREST - TUNED MODEL EVALUATION
Accuracy    : 0.8924
Precision   : 0.8699
Recall      : 0.8341
Specificity : 0.9266
F1 Score    : 0.8517
ROC-AUC     : 0.9596
PR-AUC      : 0.9409
Log Loss    : 0.2811


In [52]:
print("Number of Trees:",
      best_rf_model.n_estimators)

print("Max Depth:",
      best_rf_model.max_depth)

print("Min Samples Split:",
      best_rf_model.min_samples_split)

print("Min Samples Leaf:",
      best_rf_model.min_samples_leaf)

print("Max Features:",
      best_rf_model.max_features)

Number of Trees: 200
Max Depth: None
Min Samples Split: 2
Min Samples Leaf: 1
Max Features: None


In [53]:
random_forest_comparison = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "Specificity",
        "F1 Score",
        "ROC-AUC",
        "PR-AUC",
        "Log Loss"
    ],

    "Baseline": [
        rf_accuracy,
        rf_precision,
        rf_recall,
        rf_specificity,
        rf_f1,
        rf_roc_auc,
        rf_pr_auc,
        rf_logloss
    ],

    "Tuned": [
        rf_tuned_accuracy,
        rf_tuned_precision,
        rf_tuned_recall,
        rf_tuned_specificity,
        rf_tuned_f1,
        rf_tuned_roc_auc,
        rf_tuned_pr_auc,
        rf_tuned_logloss
    ]
})

random_forest_comparison.round(4)

,Metric,Baseline,Tuned
0,Accuracy,0.8923,0.8924
1,Precision,0.8953,0.8699
2,Recall,0.8032,0.8341
3,Specificity,0.9447,0.9266
4,F1 Score,0.8467,0.8517
5,ROC-AUC,0.9569,0.9596
6,PR-AUC,0.9394,0.9409
7,Log Loss,0.2739,0.2811


In [55]:
print("FINAL RANDOM FOREST MODEL")
print("=" * 50)

print("n_estimators     :", best_rf_model.n_estimators)
print("max_depth        :", best_rf_model.max_depth)
print("min_samples_split:", best_rf_model.min_samples_split)
print("min_samples_leaf :", best_rf_model.min_samples_leaf)
print("max_features     :", best_rf_model.max_features)

FINAL RANDOM FOREST MODEL
n_estimators     : 200
max_depth        : None
min_samples_split: 2
min_samples_leaf : 1
max_features     : None
